# Log File

## Logging Utility Function

## Log Event Function

### Description
Creates and stores log records for pipeline execution.

### Output
Appends logs to audit table for tracking and monitoring.

## Logging Utility Function

### Log Event Function
Creates and stores log records for pipeline execution.

### Output
Appends logs to audit table for monitoring.

In [0]:
from datetime import datetime

from datetime import datetime

def log_event(layer, notebook_name, status, message):
    data = [(str(layer), str(notebook_name), str(status), str(message), datetime.now())]
    
    df = spark.createDataFrame(data, ["layer", "notebook_name", "status", "message", "log_time"])
    
    df.write.mode("append").insertInto("ecom_catalog.audit.pipeline_logs")

## Audit Table Creation

### Description
Creates audit table to store pipeline execution logs.

### Structure
Includes layer, notebook name, status, message, and timestamp.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecom_catalog.audit.pipeline_logs (
    layer STRING,
    notebook_name STRING,
    status STRING,
    message STRING,
    log_time TIMESTAMP
);

### BRONZE LOG TABLE

## Logger Notebook Execution

### Description
Executes external logger notebook and defines logging function.

### Purpose
Enables logging of pipeline events into audit table.

In [0]:

dbutils.notebook.run("/Workspace/Users/bro.hitmalode@gmail.com/audit_logger", 60)


from pyspark.sql.functions import *
from datetime import datetime


def log_event(layer, notebook_name, status, message):
    data = [(layer, notebook_name, status, message, datetime.now())]
    df = spark.createDataFrame(data, ["layer", "notebook_name", "status", "message", "log_time"])
    df.write.mode("append").saveAsTable("ecom_catalog.audit.pipeline_logs")

## Bronze Data Ingestion Pipeline

### Imports and Logger Setup
Initializes required libraries and configures logging.

### Pipeline Start
Logs the start of Bronze ingestion process.

### Data Ingestion
Reads carts and products data from ADLS Bronze layer.

### Data Storage - Unity Catalog
Writes data into Bronze tables in Unity Catalog.

### Data Storage - ADLS
Stores data as Parquet files in Bronze storage.

### Error Handling
Captures and logs errors at each step of the pipeline.

In [0]:

import logging
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_logger")


log_event("BRONZE", "Bronze", "STARTED", "Bronze ingestion started")


storage_account = "finalecommercerohit"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"

try:
    carts_df = spark.read \
        .format("json") \
        .option("multiLine", "true") \
        .load(f"{bronze_base_path}/carts/")

    logger.info("Carts data loaded successfully")
    log_event("BRONZE", "Bronze", "SUCCESS", "Carts loaded")

except Exception as e:
    logger.error(f"Error reading carts data: {str(e)}")
    log_event("BRONZE", "Bronze", "FAILED", str(e))
    raise e


try:
    products_df = spark.read \
        .format("json") \
        .option("multiLine", "true") \
        .load(f"{bronze_base_path}/Products/")

    logger.info("Products data loaded successfully")
    log_event("BRONZE", "Bronze", "SUCCESS", "Products loaded")

except Exception as e:
    logger.error(f"Error reading products data: {str(e)}")
    log_event("BRONZE", "Bronze", "FAILED", str(e))
    raise e


try:
    carts_df.write.mode("append").saveAsTable("ecom_catalog.bronze.carts_bronze")
    products_df.write.mode("append").saveAsTable("ecom_catalog.bronze.products_bronze")

    logger.info("Bronze tables created in Unity Catalog")
    log_event("BRONZE", "Bronze", "SUCCESS", "Unity tables created")

except Exception as e:
    logger.error(f"Error writing Unity tables: {str(e)}")
    log_event("BRONZE", "Bronze", "FAILED", str(e))
    raise e


try:
    carts_df.write.mode("overwrite").parquet(f"{bronze_base_path}tables/carts_bronze")
    products_df.write.mode("overwrite").parquet(f"{bronze_base_path}tables/products_bronze")

    logger.info("Data also stored in ADLS Bronze zone")
    log_event("BRONZE", "Bronze", "SUCCESS", "ADLS data stored")

except Exception as e:
    logger.error(f"Error writing ADLS data: {str(e)}")
    log_event("BRONZE", "Bronze", "FAILED", str(e))
    raise e

In [0]:
%sql
SELECT * FROM ecom_catalog.audit.pipeline_logs
ORDER BY log_time DESC;